# 02 · Data integration and preprocessing (Module 4)

Builds **Dataset C** (`data/processed/solar_model_training.csv`) from:

- **Dataset A**: `nasa_solar_history.csv` (NASA POWER, LST daily, 2001 →)
- **Dataset B**: `archived_forecasts.csv` (Open-Meteo Previous Runs, `ecmwf_ifs025`, IST days, leads 1–3)

The logic lives in `src/preprocessing/`, so the live prediction engine applies exactly the same transformations. This notebook runs those functions and shows the checks.

Fitted transforms (one-hot encoding, scaling) are **not** fitted here. They go into the model pipeline in Module 6 and are fitted on the training split only.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.3f}".format)

from src.preprocessing.clean_data import load_nasa, load_forecasts, clean_nasa, clean_forecasts, FORECAST_FEATURES
from src.preprocessing.build_training_data import (
    load_config, build_climatology, build_training_table, FEATURES, TARGET, RESIDUAL_TARGET, SPLITS)

cfg = load_config()
PROCESSED = ROOT / "data" / "processed"
OUT = ROOT / "results" / "preprocessing"; OUT.mkdir(parents=True, exist_ok=True)
print("forecast model:", cfg["forecast_model"])
print("splits:", {k: cfg["splits"][k] for k in SPLITS})

forecast model: ecmwf_ifs025
splits: {'train': ['2024-03-01', '2025-05-03'], 'validation': ['2025-05-04', '2025-11-03'], 'test': ['2025-11-04', '2026-11-03']}


## 1. Load the raw processed datasets

In [2]:
nasa_raw = load_nasa()
fc_raw = load_forecasts()
print("NASA     ", nasa_raw.shape, nasa_raw["date"].min().date(), "→", nasa_raw["date"].max().date())
print("forecasts", fc_raw.shape, fc_raw["target_date"].min().date(), "→", fc_raw["target_date"].max().date())
fc_raw.head(3)

NASA      (37752, 12) 2001-01-01 → 2026-11-03
forecasts (11808, 18) 2024-03-02 → 2026-11-10


,forecast_issue_date,target_date,lead_days,region_id,latitude,longitude,grid_latitude,grid_longitude,forecast_model,forecast_shortwave_radiation,forecast_temperature_mean,forecast_temperature_max,forecast_temperature_min,forecast_humidity_mean,forecast_cloud_cover_mean,forecast_precipitation_sum,forecast_wind_speed_mean,hours_complete
0,2024-03-01,2024-03-02,1,karaikal,10.920,79.820,11.000,79.750,ecmwf_ifs025,NaN,26.721,32.000,22.900,75.875,24.583,0.000,7.808,0
1,2024-02-29,2024-03-02,2,karaikal,10.920,79.820,11.000,79.750,ecmwf_ifs025,NaN,26.562,31.600,22.800,78.542,23.250,0.000,8.108,0
2,2024-02-28,2024-03-02,3,karaikal,10.920,79.820,11.000,79.750,ecmwf_ifs025,NaN,26.662,31.700,22.900,NaN,9.708,0.000,8.392,0


## 2. Clean NASA POWER (targets)
Drop duplicate (region, date) rows and missing or out-of-range targets (outside 0–10 kWh/m²/day).

In [3]:
nasa, nasa_report = clean_nasa(nasa_raw)
nasa_report

{'rows_in': 37752,
 'duplicates_dropped': 0,
 'invalid_target_dropped': 0,
 'rows_out': 37752}

## 3. Clean archived forecasts

- **Clip to physical bounds.** Precipitation, radiation and wind must be ≥ 0, and humidity and cloud cover must be within 0–100 %. ECMWF precipitation can dip slightly below zero, which is a numerical artefact.
- **Drop incomplete forecast days.** A day is incomplete if any hour is missing for any variable. We drop these rather than impute them, because they come from archive outages.

In [4]:
fc, fc_report = clean_forecasts(fc_raw)
print({k: v for k, v in fc_report.items() if k != "incomplete_dates"})
print("incomplete dates dropped:", fc_report["incomplete_dates"])

{'rows_in': 11808, 'values_clipped': {'forecast_shortwave_radiation': 0, 'forecast_humidity_mean': 0, 'forecast_cloud_cover_mean': 0, 'forecast_precipitation_sum': 1, 'forecast_wind_speed_mean': 0}, 'duplicates_dropped': 0, 'incomplete_rows_dropped': 160, 'rows_out': 11648}
incomplete dates dropped: ['2024-03-02', '2024-03-03', '2024-03-04', '2024-03-05', '2024-03-06', '2024-03-07', '2024-03-08', '2024-03-09', '2026-04-17', '2026-04-18', '2026-04-19', '2026-04-20', '2026-04-21', '2026-04-22', '2026-04-23']


## 4. Coordinate and grid-cell correspondence

Both APIs are queried at the same requested coordinates (`config/locations.json`), but each snaps them to its own grid:

- **Open-Meteo / ECMWF IFS:** the cell is reported by the API, on a 0.25° grid.
- **NASA POWER solar (CERES SYN1deg):** a 1° grid. The cell is derived as ⌊lat⌋, ⌊lon⌋.

The check below verifies that no two regions share a NASA cell. If two did, their targets would be identical and could not be treated as independent. All taluks *within* one region share that region's cell, which the dashboard must disclose.

In [5]:
loc = json.loads((ROOT / "config" / "locations.json").read_text(encoding="utf-8"))["regions"]
cells = pd.DataFrame(loc)[["region_id", "latitude", "longitude"]]
cells["nasa_solar_cell"] = cells.apply(lambda r: f"{int(np.floor(r.latitude))}°N {int(np.floor(r.longitude))}°E", axis=1)
om = fc.groupby("region_id")[["grid_latitude", "grid_longitude"]].first()
cells = cells.join(om, on="region_id").rename(columns={"grid_latitude": "openmeteo_lat", "grid_longitude": "openmeteo_lon"})
display(cells)
assert cells["nasa_solar_cell"].is_unique, "two regions share a NASA solar grid cell"

# empirical check: share of days on which two regions have exactly the same NASA value
w = nasa.pivot(index="date", columns="region_id", values="ALLSKY_SFC_SW_DWN")
same = pd.DataFrame({a: {b: float((w[a] == w[b]).mean()) for b in w} for a in w})
print("share of identical daily NASA values between regions:")
same

,region_id,latitude,longitude,nasa_solar_cell,openmeteo_lat,openmeteo_lon
0,puducherry,11.930,79.800,11°N 79°E,12.000,79.750
1,karaikal,10.920,79.820,10°N 79°E,11.000,79.750
2,mahe,11.700,75.540,11°N 75°E,11.750,75.500
3,yanam,16.730,82.210,16°N 82°E,16.750,82.250


share of identical daily NASA values between regions:


,karaikal,mahe,puducherry,yanam
karaikal,1.000,0.000,0.000,0.000
mahe,0.000,1.000,0.000,0.000
puducherry,0.000,0.000,1.000,0.000
yanam,0.000,0.000,0.000,1.000


## 5. Region-specific climatology (leakage-safe)

For each region and day of year, this computes the mean NASA all-sky and clear-sky radiation over **2001–2023**, which is strictly before the training period starts (March 2024). The result is smoothed with a circular ±15-day moving average.

It is saved to `data/processed/climatology.csv`. This lookup table is reused unchanged at inference time.

In [6]:
c = cfg["climatology"]
assert c["end_year"] < pd.Timestamp(cfg["splits"]["train"][0]).year, "climatology overlaps training period"
clim = build_climatology(nasa, c["start_year"], c["end_year"], c["smoothing_window_days"])
clim.to_csv(PROCESSED / "climatology.csv", index=False)
print(clim.shape, "| years per day-of-year:", clim["n_years"].min(), "–", clim["n_years"].max())
clim.groupby("region_id")[["climatology_solar_radiation", "clear_sky_radiation"]].agg(["min", "mean", "max"])

(1460, 5) | years per day-of-year: 23 – 28


climatology_solar_radiation             clear_sky_radiation            
                                   min  mean   max                 min  mean   max
region_id                                                                         
karaikal                         3.942 5.399 6.397               5.553 6.478 7.045
mahe                             3.909 5.220 6.468               5.615 6.507 6.988
puducherry                       3.970 5.377 6.481               5.440 6.449 7.038
yanam                            4.236 5.012 6.429               4.831 6.189 7.036

## 6. Join, add features, assign chronological splits

- **Join:** forecasts ⨝ NASA on (`region_id`, `target_date`).
- **Features:** cyclical `doy_sin` and `doy_cos`, plus the climatology features.
- **Targets:** `actual_solar_radiation` (NASA), and the optional residual target `target_residual` = actual − forecast shortwave.
- **Split:** by `target_date`, so all three lead rows of a date always fall in the same split.

In [7]:
train_df = build_training_table(fc, nasa, clim, cfg["splits"])
train_df.to_csv(PROCESSED / "solar_model_training.csv", index=False, date_format="%Y-%m-%d")
print(train_df.shape)
train_df.head(3)

(11564, 24)


,target_date,forecast_issue_date,region_id,lead_days,forecast_model,latitude,longitude,grid_latitude,grid_longitude,split,forecast_shortwave_radiation,forecast_temperature_mean,forecast_temperature_max,forecast_temperature_min,forecast_humidity_mean,forecast_cloud_cover_mean,forecast_precipitation_sum,forecast_wind_speed_mean,doy_sin,doy_cos,climatology_solar_radiation,clear_sky_radiation,actual_solar_radiation,target_residual
0,2024-03-07,2024-03-06,karaikal,1,ecmwf_ifs025,10.920,79.820,11.000,79.750,train,6.893,26.812,32.000,22.300,72.583,8.333,0.000,8.025,0.914,0.406,6.242,6.862,6.165,-0.728
1,2024-03-07,2024-03-06,mahe,1,ecmwf_ifs025,11.700,75.540,11.750,75.500,train,6.657,28.558,32.500,24.900,67.208,2.167,0.000,11.942,0.914,0.406,6.468,6.858,6.813,0.156
2,2024-03-07,2024-03-06,puducherry,1,ecmwf_ifs025,11.930,79.800,12.000,79.750,train,6.668,27.392,33.600,21.900,69.583,15.708,0.000,8.583,0.914,0.406,6.279,6.834,6.330,-0.338


### Checks

In [8]:
# 1. every target date belongs to exactly one split
assert (train_df.groupby("target_date")["split"].nunique() == 1).all()
# 2. splits are chronological and non-overlapping
rng = train_df.groupby("split")["target_date"].agg(["min", "max"]).loc[list(SPLITS)]
assert rng["max"].iloc[0] < rng["min"].iloc[1] and rng["max"].iloc[1] < rng["min"].iloc[2]
# 3. no missing features or targets
assert train_df[FEATURES + [TARGET]].isna().sum().sum() == 0
# 4. one row per (region, date, lead)
assert not train_df.duplicated(["region_id", "target_date", "lead_days"]).any()
print("all checks passed")
rng

all checks passed


,min,max
split,,
train,2024-03-07,2025-05-03
validation,2025-05-04,2025-11-03
test,2025-11-04,2026-11-03


In [9]:
counts = train_df.pivot_table(index="split", columns="region_id", values="target_date", aggfunc="count").loc[list(SPLITS)]
counts["total rows"] = counts.sum(axis=1)
counts["target dates"] = train_df.groupby("split")["target_date"].nunique()
counts

region_id,karaikal,mahe,puducherry,yanam,total rows,target dates
split,,,,,,
train,1265,1265,1265,1265,5060,423
validation,552,552,552,552,2208,184
test,1074,1074,1074,1074,4296,358


In [10]:
# rows per lead: should be equal within each split unless a lead was incomplete on some day
train_df.pivot_table(index="split", columns="lead_days", values="target_date", aggfunc="count").loc[list(SPLITS)]

lead_days,1,2,3
split,,,
train,1692,1688,1680
validation,736,736,736
test,1432,1432,1432


## 7. Data-quality report

In [11]:
quality = {
    "nasa_cleaning": nasa_report,
    "forecast_cleaning": fc_report,
    "grid_cells": cells.to_dict(orient="records"),
    "climatology": {"years": [c["start_year"], c["end_year"]], "smoothing_window_days": c["smoothing_window_days"]},
    "training_table": {
        "rows": len(train_df),
        "features": FEATURES, "target": TARGET, "residual_target": RESIDUAL_TARGET,
        "splits": {s: {"from": str(rng.loc[s, "min"].date()), "to": str(rng.loc[s, "max"].date()),
                       "rows": int((train_df["split"] == s).sum()),
                       "dates": int(train_df.loc[train_df["split"] == s, "target_date"].nunique())}
                   for s in SPLITS},
    },
}
(OUT / "data_quality.json").write_text(json.dumps(quality, indent=2, default=str), encoding="utf-8")
print("saved", (OUT / "data_quality.json").relative_to(ROOT))

saved results\preprocessing\data_quality.json
